In [ ]:
%load_ext autoreload
%autoreload 2
import os
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

import socket
if 'trace' in socket.gethostname():
    base_fp = '/trace/group/rounce/cvwilson/Output/'
    home_fp = '/trace/home/cvwilson/research/'
else:
    base_fp = 'C:/Users/cvw30/Research/Output/'
    home_fp = 'C:/Users/cvw30/Research/'

colors = ['#63c4c7','#fcc02e','#4D559C','#60C252','#BF1F6A',
              '#F77808','#298282','#999999','#FF89B0','#427801']

In [ ]:
class Data():
    def init(self, glac_no, site):
        self.glac_no = glac_no 

        # open dataframes
        metadata_df = pd.read_csv(home_fp + 'PEBSI/data/glacier_metadata.csv', index_col=0)
        wgms_df = pd.read_csv(home_fp + 'data/wgms/data/mass_balance_point.csv', parse_dates=True)
        benchmark_fp = home_fp + 'MB_data/'
        glacier_fp = home_fp + 'PEBSI/data/by_glacier/'

        # find site attributes used in the model run 
        self.name = metadata_df[glac_no, 'name']
        self.site = site 
        site_df = pd.read_csv(glacier_fp + f'{self.name}/site_constants.csv', index_col='site')
        model_lat = site_df.loc[site, 'lat']
        model_lon = site_df.loc[site, 'lon']

        # types of data we have
        benchmark_glaciers = [f.lower() for f in os.listdir(benchmark_fp)]
        wgms_glaciers = wgms_df['glacier_name'].unique().values

        # determine which type of data we have and put it in standard format
        if self.name.upper() in wgms_glaciers:
            # WGMS glacier
            name_fmtd = self.name.upper().replace('_',' ')
            df = wgms_df.loc[wgms_df['glacier_name'] == name_fmtd]
            self.df = df.loc[self.df['original_id'] == site]

            self.period_starts = self.df['begin_date']
            self.period_ends = self.df['end_date']
            self.data = self.df['balance']
            
        elif self.name.replace('_','') in benchmark_glaciers:
            # BENCHMARK glacier
            name_fmtd = sum([f.capitalize() for f in self.name.split('_')])
            data_fn = benchmark_fp + f'{name_fmtd}/Input_{name_fmtd}_Glaciological_Data.csv'
            self.df = pd.read_csv(data_fn, parse_dates=True)
            self.df = self.df.loc[self.df['site_name'] == site]
            self.period_starts = self.df['spring_date']

        return 
    
    def get_seasonal_mb(self, ds):
        # grab only periods within the dataset
        valid_periods = np.where((self.period_starts >= ds.time.values[0]) & 
                                 (self.period_ends <= ds.time.values[-1]))[0]
        period_starts = self.period_starts[valid_periods]
        period_ends = self.period_ends[valid_periods]
        meas = self.data[valid_periods]

        mod = []
        for start, end in zip(period_starts, period_ends):
            mb_mod = ds.sel(time=slice(start, end)).MB.sum()
            mod.append(mb_mod)
        mod = np.array(mod)

        self.mod = mod 
        self.meas = meas
        return

    def mae(self):
        return np.mean(np.abs(self.mod - self.meas))
    def rmse(self):
        return np.sqrt(np.mean(np.square(self.mod - self.meas)))
    

In [ ]:
site = 'NWB1'
data = Data('01.01390', site)
ds = xr.open_dataset(f'../../Output/{data.name}{site}_2026_02_10_1.nc')
data.get_seasonal_mb(ds)

In [ ]:
glaciersite = 'gulkanaB'
